# Residual Signal Analysis

Signal vs noise in the residual stream: SNR tracking,
component interference, update coherence, and orthogonality.

## Why This Matters

Residual stream signal characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_signal_analysis import (
    signal_noise_ratio, component_interference,
    residual_coherence, update_orthogonality,
    residual_signal_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Signal-to-Noise Ratio

Signal = projection onto final prediction direction. Noise = orthogonal component.

In [ ]:
result = signal_noise_ratio(model, tokens, position=-1)
print(f"Final SNR: {result['final_snr']:.4f}")
for p in result['per_layer']:
    bar = '█' * int(p['snr'] * 5)
    print(f"  Layer {p['layer']}: signal={p['signal']:.4f}, noise={p['noise']:.4f}, "
          f"SNR={p['snr']:.4f}, frac={p['signal_fraction']:.4f} {bar}")

## Component Interference

Do attention and MLP outputs cancel each other?

In [ ]:
for layer in range(4):
    result = component_interference(model, tokens, layer=layer, position=-1)
    flag = ' [INTERFERENCE]' if result['has_interference'] else ''
    print(f"  Layer {layer}: cos={result['cosine']:+.4f}, "
          f"cancel={result['cancellation']:.4f}{flag}")
    print(f"    attn={result['attn_norm']:.4f}, mlp={result['mlp_norm']:.4f}, "
          f"combined={result['combined_norm']:.4f}")

## Residual Coherence

Do successive layer updates point in consistent directions?

In [ ]:
result = residual_coherence(model, tokens, position=-1)
flag = ' [COHERENT]' if result['is_coherent'] else ''
print(f"Mean coherence: {result['mean_coherence']:.4f}{flag}")
for p in result['per_pair']:
    print(f"  Layers {p['layer_pair']}: cos={p['cosine']:+.4f}")

## Update Orthogonality

Are layer updates independent (orthogonal) or redundant?

In [ ]:
result = update_orthogonality(model, tokens, position=-1)
print(f"Mean orthogonality: {result['mean_orthogonality']:.4f}")
print(f"Mean |cosine|: {result['mean_abs_cosine']:.4f}")
for p in result['pairs']:
    print(f"  Layers {p['layers']}: cos={p['cosine']:+.4f}")

## Signal Analysis Summary

In [ ]:
result = residual_signal_summary(model, tokens, position=-1)
for k, v in result.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")